In [ ]:
!pip install pandas

In [ ]:
!pip install numpy

In [ ]:
!pip install matplotlib

In [ ]:
!pip install seaborn

In [ ]:
!pip install ploty

In [ ]:
!pip install plotly

In [ ]:
!pip install notebook ipywidgets --upgrade

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

df = pd.read_csv("C:/Users/SRI TEJASWINI/Desktop/DS & A/Task2.csv")

df.columns = df.columns.str.strip()

if 'Churn' not in df.columns:
    print("Available columns:", df.columns)
    raise KeyError("Churn column not found! Please check CSV.")

df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

df = df.dropna(subset=['Churn'])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

for col in df.columns:
    if df[col].dtype == object and df[col].nunique() == 2:
        df[col] = df[col].map({'Yes':1, 'No':0})

overview_table = df.head(10).copy()
overview_table.index = np.arange(1, len(overview_table)+1)  # serial from 1

fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(overview_table.columns),
                fill_color='mediumslateblue',
                font=dict(color='white', size=12),
                align='center',
                line_color='black'),
    cells=dict(values=[overview_table[col] for col in overview_table.columns],
               fill_color='lavender',
               align='center',
               line_color='black'))
])
fig_table.update_layout(title='Sample Data Overview')
fig_table.show()

churn_colors = {0:'mediumseagreen', 1:'crimson'}
fig = px.pie(df, names='Churn', title='Overall Churn Distribution', 
             color='Churn', color_discrete_map=churn_colors,
             hole=0.3)
fig.update_traces(textinfo='percent+label')
fig.show()

fig = px.bar(df, x='Contract', color='Churn', barmode='group',
             color_discrete_map=churn_colors,
             title='Churn by Contract Type')
fig.update_layout(xaxis_title='Contract Type', yaxis_title='Number of Customers')
fig.show()

fig = px.bar(df, y='PaymentMethod', color='Churn', barmode='group',
             color_discrete_map=churn_colors,
             title='Churn by Payment Method')
fig.update_layout(yaxis_title='Payment Method', xaxis_title='Number of Customers')
fig.show()

fig = px.box(df, x='Churn', y='tenure', color='Churn',
             color_discrete_map=churn_colors,
             title='Tenure vs Churn')
fig.update_layout(xaxis_title='Churn', yaxis_title='Tenure (Months)')
fig.show()

fig = px.box(df, x='Churn', y='MonthlyCharges', color='Churn',
             color_discrete_map=churn_colors,
             title='Monthly Charges vs Churn')
fig.update_layout(xaxis_title='Churn', yaxis_title='Monthly Charges ($)')
fig.show()

print("\n--- Actionable Insights ---")
print("1. Month-to-month contracts churn more than one- or two-year contracts.")
print("2. Customers with higher monthly charges tend to churn more.")
print("3. Customers with longer tenure are more loyal.")
print("4. Target payment methods with higher churn rates for retention campaigns.")
print("5. Upselling or loyalty incentives for high-risk segments can reduce churn.")

df.to_csv("Churn_Cleaned.csv", index=False)